# Federated Learning (FedAvg) — Tomato Leaf Disease Classification

This notebook trains the proposed **SimpleNetAugDR-3** model in a **federated
learning** setting using the **Federated Averaging (FedAvg)** algorithm across
four clients, without sharing raw data between them.

**Data split (proper three-way):**

* **train/** is split (stratified) into a **client training pool** and a **validation** set.
  The pool is partitioned across the clients; the validation set is used to monitor the
  global model after each communication round.
* **valid/** is kept as a **held-out test set**, used only for the final classification
  report — it is never seen during training or model selection.

**Outputs produced:** per-client class distribution (Table I), global-model metrics per
communication round on validation (Table VI), and the final classification report on the
held-out test set (Table VIII).

## 1. Setup and imports

In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall, AUC, TopKCategoricalAccuracy
from tensorflow.keras.utils import to_categorical, normalize

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

import warnings
warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

In [ ]:
# ----------------------------- Configuration -------------------------------
SEED = 123
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Dataset paths (PlantVillage "New Plant Diseases Dataset (Augmented)").
TRAIN_DIR = './new-plant-diseases-dataset/train/'
VALID_DIR = './new-plant-diseases-dataset/valid/'

# Federated-learning protocol.
NUM_CLIENTS  = 4          # number of federated clients
NUM_ROUNDS   = 5          # communication rounds (adopted in the paper)
LOCAL_EPOCHS = 1          # local epochs per client per round
BATCH_SIZE   = 16
LR           = 0.001
VAL_SPLIT    = 0.2        # fraction of train/ held out as the validation split
IMG_SIZE     = (128, 128)

CLASS_NAMES = [
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
]
NUM_CLASSES = len(CLASS_NAMES)

OUT_DIR = 'fl_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Data loading — train / validation / test

`train/` is loaded and split (stratified) into a **client training pool** and a
**validation** set; `valid/` is loaded and kept as the **held-out test set**.

In [ ]:
def load_data(dir_path, img_size=IMG_SIZE):
    """Load every 'Tomato__' subfolder under dir_path into arrays."""
    X, Y, i = [], [], 0
    for path in tqdm(sorted(os.listdir(dir_path)),
                     desc=os.path.basename(dir_path.rstrip('/'))):
        if path.startswith('.') or 'Tomato__' not in path:
            continue
        sub = os.path.join(dir_path, path)
        for f in os.listdir(sub):
            fp = os.path.join(sub, f)
            if not f.startswith('.') and os.path.isfile(fp):
                img = cv2.imread(fp)
                img = Image.fromarray(img, 'RGB').resize(img_size)
                X.append(np.array(img)); Y.append(i)
        i += 1
    return np.array(X), np.array(Y)

In [ ]:
def prepare_data(img_size=IMG_SIZE, val_split=VAL_SPLIT):
    """Return (X_pool, y_pool_int, X_val, y_val, X_test, y_test).

    * X_pool / y_pool_int : client training pool (integer labels, ready to shard)
    * X_val  / y_val      : validation set (one-hot) for per-round monitoring
    * X_test / y_test     : held-out test set (one-hot) for the final report
    """
    X_all, y_all = load_data(TRAIN_DIR, img_size)     # client pool + validation
    X_te,  y_te  = load_data(VALID_DIR, img_size)     # held-out test set

    X_pool, X_val, y_pool, y_val = train_test_split(
        X_all, y_all, test_size=val_split, stratify=y_all, random_state=SEED)

    X_pool = normalize(X_pool.astype(np.float32), axis=1)
    X_val  = normalize(X_val.astype(np.float32),  axis=1)
    X_te   = normalize(X_te.astype(np.float32),   axis=1)

    print(f'Client pool: {X_pool.shape} | Val: {X_val.shape} | Test (held-out): {X_te.shape}')
    return (X_pool, y_pool,
            X_val, to_categorical(y_val, NUM_CLASSES),
            X_te,  to_categorical(y_te,  NUM_CLASSES))


X_pool, y_pool, X_val, y_val, X_test, y_test = prepare_data()

## 3. Split the training pool across clients

The pool is partitioned across `NUM_CLIENTS` clients **stratified by class**, so each
client holds a balanced slice of all ten classes (the IID setting). The per-client class
distribution below corresponds to Table I.

In [ ]:
def make_clients(X, y_int, num_clients, seed=SEED):
    """Stratified partition of (X, y_int) into num_clients balanced shards."""
    rng = np.random.default_rng(seed)
    client_idx = [[] for _ in range(num_clients)]
    for c in np.unique(y_int):
        idx = np.where(y_int == c)[0]
        rng.shuffle(idx)
        for k, part in enumerate(np.array_split(idx, num_clients)):
            client_idx[k].extend(part.tolist())
    clients = []
    for k in range(num_clients):
        ix = np.array(client_idx[k]); rng.shuffle(ix)
        clients.append((X[ix], to_categorical(y_int[ix], NUM_CLASSES)))
    return clients


client_data = make_clients(X_pool, y_pool, NUM_CLIENTS)

# Table I — per-client class distribution.
dist = {f'Client {k+1}': Counter(np.argmax(yc, axis=1)) for k, (_, yc) in enumerate(client_data)}
table1 = pd.DataFrame({cl: [dist[cl].get(c, 0) for c in range(NUM_CLASSES)] for cl in dist},
                      index=[f'C{c}' for c in range(NUM_CLASSES)])
table1['Total'] = table1.sum(axis=1)
table1.loc['Total'] = table1.sum(axis=0)
table1.to_csv(os.path.join(OUT_DIR, 'table1_client_distribution.csv'))
table1

## 4. Model — SimpleNetAugDR-3

The same architecture used in the benchmark notebook (three conv blocks 64/64/64, dropout 0.5).

In [ ]:
def create_model(num_classes=NUM_CLASSES, lr=LR):
    model = Sequential([
        Conv2D(64, (4, 4), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (4, 4), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),   # 64/64/64 -> matches Figure 2 & benchmark NB01
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=lr),
        metrics=['accuracy', Precision(name='precision'), Recall(name='recall'),
                 AUC(name='auc'), TopKCategoricalAccuracy(k=3, name='top_k_categorical_accuracy')])
    return model


print(f'Parameters: {create_model().count_params():,}  (SimpleNetAugDR-3, 64/64/64)')

## 5. Federated training (FedAvg)

Each communication round: every client copies the current global weights, trains locally
on its own data for `LOCAL_EPOCHS`, and returns its updated weights. The server aggregates
them with the **weighted FedAvg** rule

$$ w_{t+1} = \sum_{k=1}^{K} \frac{n_k}{N}\, w_k $$

where $n_k$ is the number of samples on client $k$ and $N$ the total. After each round the
global model is evaluated on the **validation** set (Table VI). The best round (by
validation accuracy) is saved for the final test evaluation and for reuse in the analysis
notebook.

In [ ]:
def fedavg(local_weights, sizes):
    """Weighted average of client weight lists: sum_k (n_k / N) * w_k."""
    total = float(sum(sizes))
    return [sum((sizes[k] / total) * lw[layer] for k, lw in enumerate(local_weights))
            for layer in range(len(local_weights[0]))]


def evaluate(model, X, y_onehot):
    y_pred = np.argmax(model.predict(X, batch_size=BATCH_SIZE, verbose=0), axis=1)
    y_true = np.argmax(y_onehot, axis=1)
    loss = model.evaluate(X, y_onehot, verbose=0)[0]
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    return dict(Loss=round(loss, 4), Accuracy=round(acc * 100, 2),
                Precision=round(prec * 100, 2), Recall=round(rec * 100, 2),
                F1=round(f1 * 100, 2))


def federated_train(client_data, X_val, y_val, num_rounds=NUM_ROUNDS,
                    local_epochs=LOCAL_EPOCHS, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    global_model = create_model()
    sizes = [len(Xc) for Xc, _ in client_data]

    rounds, best_acc, best_path = [], -1.0, os.path.join(OUT_DIR, 'best_global_model.keras')
    for rnd in range(1, num_rounds + 1):
        global_w = global_model.get_weights()
        local_weights = []
        for cid, (Xc, yc) in enumerate(client_data):
            local = create_model()
            local.set_weights(global_w)
            local.fit(Xc, yc, batch_size=BATCH_SIZE, epochs=local_epochs, verbose=0)
            local_weights.append(local.get_weights())
            del local; gc.collect()

        global_model.set_weights(fedavg(local_weights, sizes))   # weighted FedAvg

        m = evaluate(global_model, X_val, y_val)                  # monitor on validation
        m = {'Round': rnd, **m}
        rounds.append(m)
        print(f"Round {rnd}/{num_rounds} | val acc {m['Accuracy']:.2f}% | "
              f"P {m['Precision']:.2f} R {m['Recall']:.2f} F1 {m['F1']:.2f}")

        if m['Accuracy'] > best_acc:
            best_acc = m['Accuracy']; global_model.save(best_path)

    return global_model, pd.DataFrame(rounds), best_path


global_model, round_df, best_path = federated_train(client_data, X_val, y_val)
round_df.to_csv(os.path.join(OUT_DIR, 'table6_round_metrics.csv'), index=False)
round_df

## 6. Final evaluation on the held-out test set

The best global model (selected by validation accuracy) is evaluated **once** on the
held-out test set. The per-class classification report corresponds to Table VIII.

In [ ]:
best_global = keras.models.load_model(best_path)

y_prob = best_global.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(f"Held-out TEST accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%\n")
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4,
                               zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(os.path.join(OUT_DIR, 'table8_test_classification_report.csv'))
report_df

In [ ]:
# Normalised confusion matrix on the held-out test set.
cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(9, 8))
plt.imshow(cm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(fraction=0.046)
ticks = range(NUM_CLASSES)
plt.xticks(ticks, [f'C{i}' for i in ticks]); plt.yticks(ticks, [f'C{i}' for i in ticks])
for i in ticks:
    for j in ticks:
        if cm[i, j] > 0.01:
            plt.text(j, i, f'{cm[i, j]*100:.0f}', ha='center', va='center', fontsize=7,
                     color='white' if cm[i, j] > 0.5 else 'black')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Global model — confusion matrix (held-out test set)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fl_test_confusion_matrix.png'), dpi=200, bbox_inches='tight')
plt.show()

## 7. Convergence curves over communication rounds

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(round_df['Round'], round_df['Accuracy'], 'o-', label='Accuracy')
ax[0].plot(round_df['Round'], round_df['Precision'], 's--', label='Precision')
ax[0].plot(round_df['Round'], round_df['Recall'], '^--', label='Recall')
ax[0].plot(round_df['Round'], round_df['F1'], 'd--', label='F1-Score')
ax[0].set_xlabel('Communication round'); ax[0].set_ylabel('Validation metric (%)')
ax[0].set_title('Global model metrics per round'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(round_df['Round'], round_df['Loss'], 'o-', color='crimson')
ax[1].set_xlabel('Communication round'); ax[1].set_ylabel('Validation loss')
ax[1].set_title('Global model loss per round'); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fl_convergence.png'), dpi=200, bbox_inches='tight')
plt.show()